# Capítulo 2: Ingeniería de Características, VSM y Similitud del Coseno

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/caracena/apunte-analitica-textual/blob/main/capitulos/clase2-ingenieria-caracteristicas-vsm.ipynb)

## Objetivos de aprendizaje

- Construir la Matriz de Término-Frecuencia (DTM) a partir de un corpus.
- Comprender y aplicar el esquema de ponderación TF-IDF.
- Representar documentos en el Vector Space Model (VSM).
- Calcular la Similitud del Coseno para comparar documentos.

## 2.1 De texto a números: la necesidad de representaciones

Los algoritmos de machine learning trabajan con números, no con texto. Necesitamos transformar los documentos en **vectores numéricos** que capturen su contenido semántico.

### Bag of Words (BoW)

El modelo más simple: representar cada documento como un vector donde cada dimensión corresponde a una palabra del vocabulario, y el valor es la frecuencia de aparición.

**Limitaciones**:
- Ignora el orden de las palabras.
- No captura relaciones semánticas.
- Genera matrices muy dispersas (*sparse*).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Corpus de ejemplo
corpus = [
    "el gato se sentó en la alfombra",
    "el perro se sentó en el sofá",
    "el gato y el perro jugaron juntos",
    "la alfombra es nueva y suave"
]

# Crear la Matriz de Término-Frecuencia
vectorizer = CountVectorizer()
dtm = vectorizer.fit_transform(corpus)

# Visualizar como DataFrame
df_dtm = pd.DataFrame(
    dtm.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(corpus))]
)
print("Matriz de Término-Frecuencia (DTM):")
df_dtm

## 2.2 TF-IDF: Term Frequency - Inverse Document Frequency

El esquema **TF-IDF** pondera la importancia de un término en un documento relativo a todo el corpus.

### Fórmula

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$

Donde:

$$\text{TF}(t, d) = \frac{\text{frecuencia de } t \text{ en } d}{\text{total de términos en } d}$$

$$\text{IDF}(t) = \log\left(\frac{N}{\text{df}(t)}\right)$$

- $N$: número total de documentos.
- $\text{df}(t)$: número de documentos que contienen el término $t$.

### Intuición

- **TF alto**: El término aparece frecuentemente en el documento → es relevante para ese documento.
- **IDF alto**: El término aparece en pocos documentos → es un buen discriminador.
- Palabras comunes como "el", "de", "en" tendrán IDF bajo porque aparecen en muchos documentos.

In [ ]:
# Cálculo manual de TF-IDF
def calcular_tf(documento):
    """Calcula la frecuencia de término normalizada."""
    palabras = documento.lower().split()
    tf = {}
    for palabra in palabras:
        tf[palabra] = tf.get(palabra, 0) + 1
    # Normalizar por el total de palabras
    total = len(palabras)
    return {k: v / total for k, v in tf.items()}

def calcular_idf(corpus):
    """Calcula el IDF para cada término del corpus."""
    N = len(corpus)
    idf = {}
    # Obtener vocabulario
    vocabulario = set()
    for doc in corpus:
        vocabulario.update(doc.lower().split())
    # Calcular df para cada término
    for termino in vocabulario:
        df = sum(1 for doc in corpus if termino in doc.lower().split())
        idf[termino] = np.log(N / df)
    return idf

# Ejemplo
tf_doc1 = calcular_tf(corpus[0])
idf = calcular_idf(corpus)

print("TF del Doc 1:")
for t, v in sorted(tf_doc1.items()):
    print(f"  {t}: TF={v:.3f}, IDF={idf[t]:.3f}, TF-IDF={v * idf[t]:.3f}")

In [ ]:
# TF-IDF con scikit-learn
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

df_tfidf = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(corpus))]
)
print("Matriz TF-IDF:")
df_tfidf.round(3)

## 2.3 Vector Space Model (VSM)

El **Vector Space Model** representa documentos como vectores en un espacio multidimensional, donde cada dimensión corresponde a un término del vocabulario.

### Propiedades

- Cada documento es un punto (o vector) en un espacio de $|V|$ dimensiones.
- La dirección del vector indica el "tema" del documento.
- La magnitud del vector indica la "fuerza" o relevancia del contenido.
- Documentos similares apuntan en direcciones similares.

In [ ]:
import matplotlib.pyplot as plt

# Ejemplo simplificado en 2D: solo usamos 2 términos
corpus_simple = [
    "gato gato perro",
    "perro perro perro",
    "gato gato gato"
]

vec_simple = CountVectorizer()
X = vec_simple.fit_transform(corpus_simple).toarray()
palabras = vec_simple.get_feature_names_out()

fig, ax = plt.subplots(1, 1, figsize=(6, 6))
colores = ['#e74c3c', '#3498db', '#2ecc71']
for i, (x, y) in enumerate(X):
    ax.annotate('', xy=(x, y), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=colores[i], lw=2))
    ax.annotate(f'Doc {i+1}', xy=(x, y), fontsize=12,
                ha='left', va='bottom', color=colores[i])

ax.set_xlim(-0.5, 4)
ax.set_ylim(-0.5, 4)
ax.set_xlabel(palabras[0], fontsize=14)
ax.set_ylabel(palabras[1], fontsize=14)
ax.set_title('Documentos en el Vector Space Model', fontsize=14)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 2.4 Similitud del Coseno

La **Similitud del Coseno** mide el ángulo entre dos vectores, independientemente de su magnitud. Es la métrica más utilizada para comparar documentos en el VSM.

### Fórmula

$$\text{sim}(\mathbf{a}, \mathbf{b}) = \cos(\theta) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \|\mathbf{b}\|} = \frac{\sum_{i=1}^{n} a_i b_i}{\sqrt{\sum_{i=1}^{n} a_i^2} \sqrt{\sum_{i=1}^{n} b_i^2}}$$

### Interpretación

| Valor | Significado |
|-------|-------------|
| 1 | Documentos idénticos (misma dirección) |
| 0 | Documentos ortogonales (sin relación) |
| -1 | Documentos opuestos (raro en texto, ya que los valores suelen ser positivos) |

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Calcular similitud del coseno entre todos los documentos
similitudes = cosine_similarity(tfidf_matrix)

df_sim = pd.DataFrame(
    similitudes,
    columns=[f"Doc {i+1}" for i in range(len(corpus))],
    index=[f"Doc {i+1}" for i in range(len(corpus))]
)
print("Matriz de Similitud del Coseno:")
df_sim.round(3)

In [ ]:
# Implementación manual de la similitud del coseno
def similitud_coseno(vec_a, vec_b):
    """Calcula la similitud del coseno entre dos vectores."""
    dot_product = np.dot(vec_a, vec_b)
    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot_product / (norm_a * norm_b)

# Comparar Doc 1 y Doc 2
vec1 = tfidf_matrix[0].toarray().flatten()
vec2 = tfidf_matrix[1].toarray().flatten()

sim = similitud_coseno(vec1, vec2)
print(f"Similitud entre Doc 1 y Doc 2: {sim:.4f}")
print(f"  Doc 1: '{corpus[0]}'")
print(f"  Doc 2: '{corpus[1]}'")

## 2.5 Aplicación: Motor de búsqueda simple

Usemos TF-IDF y similitud del coseno para construir un buscador básico.

In [ ]:
# Motor de búsqueda con TF-IDF
documentos = [
    "Python es un lenguaje de programación versátil y fácil de aprender",
    "El análisis de datos con pandas permite manipular grandes volúmenes de información",
    "Machine learning es una rama de la inteligencia artificial",
    "Los modelos de lenguaje natural procesan texto automáticamente",
    "La visualización de datos con matplotlib genera gráficos informativos",
    "Deep learning utiliza redes neuronales profundas para aprender representaciones",
    "El procesamiento de lenguaje natural incluye análisis de sentimiento",
    "Scikit-learn ofrece algoritmos de clasificación y regresión"
]

# Ajustar el vectorizador con los documentos
tfidf = TfidfVectorizer()
matriz_docs = tfidf.fit_transform(documentos)

def buscar(query, top_k=3):
    """Busca los documentos más relevantes para una consulta."""
    query_vec = tfidf.transform([query])
    similitudes = cosine_similarity(query_vec, matriz_docs).flatten()
    indices = similitudes.argsort()[::-1][:top_k]
    
    print(f"Búsqueda: '{query}'\n")
    for rank, idx in enumerate(indices, 1):
        if similitudes[idx] > 0:
            print(f"  {rank}. (sim={similitudes[idx]:.3f}) {documentos[idx]}")

buscar("inteligencia artificial y aprendizaje")
print()
buscar("análisis y visualización de datos")

## 2.6 Otras métricas de distancia

Además de la similitud del coseno, existen otras métricas útiles:

| Métrica | Fórmula | Uso típico |
|---------|---------|------------|
| **Euclidiana** | $d = \sqrt{\sum(a_i - b_i)^2}$ | Sensible a la magnitud |
| **Manhattan** | $d = \sum |a_i - b_i|$ | Espacios de alta dimensión |
| **Jaccard** | $J = \frac{|A \cap B|}{|A \cup B|}$ | Conjuntos de palabras |

In [ ]:
from sklearn.metrics.pairwise import euclidean_distances

# Comparar métricas
print("Comparación de métricas entre Doc 1 y Doc 2:")
print(f"  Similitud Coseno:   {cosine_similarity(tfidf_matrix[0], tfidf_matrix[1])[0][0]:.4f}")
print(f"  Distancia Euclidiana: {euclidean_distances(tfidf_matrix[0], tfidf_matrix[1])[0][0]:.4f}")

# Similitud de Jaccard
set1 = set(corpus[0].split())
set2 = set(corpus[1].split())
jaccard = len(set1 & set2) / len(set1 | set2)
print(f"  Similitud Jaccard:  {jaccard:.4f}")

## Resumen

En este capítulo aprendimos:

- **Bag of Words**: Representación simple basada en frecuencias de términos.
- **TF-IDF**: Pondera la importancia de los términos considerando su frecuencia en el documento y su rareza en el corpus.
- **Vector Space Model**: Cada documento es un vector en un espacio de alta dimensionalidad.
- **Similitud del Coseno**: Mide la similitud entre documentos basándose en el ángulo entre sus vectores.

En el próximo capítulo usaremos estas representaciones para **agrupar documentos** automáticamente mediante algoritmos de clustering.